In [1]:
from pathlib import Path
import json
import numpy as np
import xarray as xr
from netCDF4 import Dataset


In [2]:
def safe_str(x):
    try:
        if isinstance(x, bytes):
            return x.decode("utf-8", "ignore")
        return str(x)
    except Exception:
        return repr(x)

def decode_char_array(arr):
    try:
        arr = np.asarray(arr)
        if arr.dtype.kind in ("S", "a", "O", "U"):
            if arr.ndim > 1 and arr.shape[-1] <= 1024:
                try:
                    joined = np.apply_along_axis(
                        lambda row: b"".join([
                            c if isinstance(c, (bytes, bytearray)) else bytes(str(c), "utf-8")
                            for c in row
                        ]).decode("utf-8", "ignore"),
                        axis=-1,
                        arr=arr
                    )
                    return joined
                except Exception:
                    pass
            vectorized = np.vectorize(lambda v: safe_str(v))
            return vectorized(arr)
        else:
            return arr
    except Exception:
        return arr


In [3]:
def inventory_nc(path):
    inv = {
        "path": str(path),
        "dimensions": {},
        "global_attributes": {},
        "variables": {}
    }

    try:
        ds = xr.open_dataset(path, decode_cf=True, decode_times=False)
        inv["dimensions"] = dict(ds.dims)
        inv["global_attributes"] = {k: safe_str(v) for k, v in ds.attrs.items()}

        for name in ds.variables:
            var = ds.variables[name]
            try:
                vals = var.values
                vals = decode_char_array(vals)
                if hasattr(vals, "size") and vals.size > 20:
                    sample = vals.flatten()[:20].tolist()
                else:
                    sample = vals.tolist() if hasattr(vals, "tolist") else safe_str(vals)
            except Exception as e:
                sample = f"<unreadable: {e}>"

            inv["variables"][name] = {
                "dtype": str(var.dtype),
                "shape": tuple(var.shape),
                "attributes": {k: safe_str(v) for k, v in var.attrs.items()},
                "sample": sample
            }

        ds.close()
        return inv

    except Exception:
        # fallback to netCDF4
        ds = Dataset(path)
        inv["dimensions"] = {k: len(ds.dimensions[k]) for k in ds.dimensions}
        inv["global_attributes"] = {a: safe_str(getattr(ds, a)) for a in ds.ncattrs()}

        for name in ds.variables:
            v = ds.variables[name]
            try:
                vals = v[:]
                vals = decode_char_array(vals)
                if hasattr(vals, "size") and vals.size > 20:
                    sample = np.asarray(vals).flatten()[:20].tolist()
                else:
                    sample = vals.tolist() if hasattr(vals, "tolist") else safe_str(vals)
            except Exception as e:
                sample = f"<unreadable: {e}>"

            inv["variables"][name] = {
                "dtype": str(v.datatype),
                "shape": tuple(v.shape),
                "attributes": {a: safe_str(getattr(v, a)) for a in v.ncattrs()},
                "sample": sample
            }

        ds.close()
        return inv


In [4]:
# Path to your base data directory
base_path = Path(r"c:\Users\Admin\Documents\PROJECTS\SIH\data")

# Subfolder for inventory JSON files
inventory_folder = base_path / "inventories"
inventory_folder.mkdir(parents=True, exist_ok=True)

# NetCDF files to process
nc_files = [
    base_path / "1900121_meta.nc",
    base_path / "1900121_prof.nc",
    base_path / "1900121_Rtraj.nc",
    base_path / "1900121_tech.nc"
]


In [5]:
for fn in nc_files:
    if not fn.exists():
        print(f"❌ File not found: {fn.name}")
        continue

    print(f"📥 Processing: {fn.name}")
    inv = inventory_nc(fn)

    # Output path inside the inventories folder
    out = inventory_folder / f"{fn.stem}_inventory.json"
    out.write_text(json.dumps(inv, indent=2, ensure_ascii=False))

    print(f"✅ Saved inventory to: {out.name}")
    print("  ➤ Dimensions:", inv["dimensions"])
    print("  ➤ Variables:", list(inv["variables"].keys()))
    print("  ➤ Global Attributes:", list(inv["global_attributes"].keys()))
    print("-" * 60)


📥 Processing: 1900121_meta.nc


C:\Users\Admin\AppData\Local\Temp\ipykernel_7108\4275105235.py:11: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  inv["dimensions"] = dict(ds.dims)
C:\Users\Admin\AppData\Local\Temp\ipykernel_7108\4275105235.py:11: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  inv["dimensions"] = dict(ds.dims)


✅ Saved inventory to: 1900121_meta_inventory.json
  ➤ Dimensions: {'N_TRANS_SYSTEM': 1, 'N_POSITIONING_SYSTEM': 1, 'N_LAUNCH_CONFIG_PARAM': 14, 'N_CONFIG_PARAM': 14, 'N_MISSIONS': 1, 'N_SENSOR': 3, 'N_PARAM': 3}
  ➤ Variables: ['DATA_TYPE', 'FORMAT_VERSION', 'HANDBOOK_VERSION', 'DATE_CREATION', 'DATE_UPDATE', 'PLATFORM_NUMBER', 'PTT', 'TRANS_SYSTEM', 'TRANS_SYSTEM_ID', 'TRANS_FREQUENCY', 'POSITIONING_SYSTEM', 'PLATFORM_FAMILY', 'PLATFORM_TYPE', 'PLATFORM_MAKER', 'FIRMWARE_VERSION', 'MANUAL_VERSION', 'FLOAT_SERIAL_NO', 'STANDARD_FORMAT_ID', 'DAC_FORMAT_ID', 'WMO_INST_TYPE', 'PROJECT_NAME', 'DATA_CENTRE', 'PI_NAME', 'ANOMALY', 'BATTERY_TYPE', 'BATTERY_PACKS', 'CONTROLLER_BOARD_TYPE_PRIMARY', 'CONTROLLER_BOARD_TYPE_SECONDARY', 'CONTROLLER_BOARD_SERIAL_NO_PRIMARY', 'CONTROLLER_BOARD_SERIAL_NO_SECONDARY', 'SPECIAL_FEATURES', 'FLOAT_OWNER', 'OPERATING_INSTITUTION', 'CUSTOMISATION', 'LAUNCH_DATE', 'LAUNCH_LATITUDE', 'LAUNCH_LONGITUDE', 'LAUNCH_QC', 'START_DATE', 'START_DATE_QC', 'STARTUP_DATE

C:\Users\Admin\AppData\Local\Temp\ipykernel_7108\4275105235.py:11: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  inv["dimensions"] = dict(ds.dims)
C:\Users\Admin\AppData\Local\Temp\ipykernel_7108\4275105235.py:11: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  inv["dimensions"] = dict(ds.dims)


✅ Saved inventory to: 1900121_Rtraj_inventory.json
  ➤ Dimensions: {'N_PARAM': 3, 'N_MEASUREMENT': 1847, 'N_CYCLE': 99, 'N_HISTORY': 1}
  ➤ Variables: ['DATE_CREATION', 'DATE_UPDATE', 'PLATFORM_NUMBER', 'DATA_CENTRE', 'WMO_INST_TYPE', 'PROJECT_NAME', 'PI_NAME', 'DATA_TYPE', 'FORMAT_VERSION', 'HANDBOOK_VERSION', 'REFERENCE_DATE_TIME', 'POSITIONING_SYSTEM', 'TRAJECTORY_PARAMETERS', 'DATA_STATE_INDICATOR', 'PLATFORM_TYPE', 'FLOAT_SERIAL_NO', 'FIRMWARE_VERSION', 'JULD', 'JULD_STATUS', 'JULD_QC', 'JULD_ADJUSTED', 'JULD_ADJUSTED_STATUS', 'JULD_ADJUSTED_QC', 'LATITUDE', 'LONGITUDE', 'POSITION_ACCURACY', 'POSITION_QC', 'CYCLE_NUMBER', 'CYCLE_NUMBER_ADJUSTED', 'MEASUREMENT_CODE', 'PRES', 'PRES_QC', 'PRES_ADJUSTED', 'PRES_ADJUSTED_QC', 'PRES_ADJUSTED_ERROR', 'TEMP', 'TEMP_QC', 'TEMP_ADJUSTED', 'TEMP_ADJUSTED_QC', 'TEMP_ADJUSTED_ERROR', 'PSAL', 'PSAL_QC', 'PSAL_ADJUSTED', 'PSAL_ADJUSTED_QC', 'PSAL_ADJUSTED_ERROR', 'AXES_ERROR_ELLIPSE_MAJOR', 'AXES_ERROR_ELLIPSE_MINOR', 'AXES_ERROR_ELLIPSE_ANGLE',